In [48]:
import torch 
import numpy as np
from sklearn.metrics import f1_score
import os
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import random
import timm
import tqdm
import torch.nn as nn
from PIL import Image

In [49]:
# 2. Thiết lập Seed và Transform

# Thiết lập Seed để tái lập kết quả
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"Đã thiết lập seed = {seed}")

SEED = 42
set_seed(SEED)

# Chuẩn hóa ảnh theo chuẩn ImageNet (mean/std)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])
print("Đã định nghĩa val_transform!")

Đã thiết lập seed = 42
Đã định nghĩa val_transform!


In [50]:
# 3. Định nghĩa Dataset

class TrashDataset(Dataset):
    """
    Dataset cho phân loại rác TrashNet 6 classes
    Labels gốc trong CSV: 1-6 (cardboard, glass, metal, paper, plastic, trash)
    PyTorch Labels: 0-5
    """
    def __init__(self, root_dir, csv_file_name=None, transform=None, is_test=False):
        self.root_dir = root_dir
        self.img_dir = os.path.join(root_dir, "images")
        self.transform = transform
        self.is_test = is_test

        if is_test:
            # Test set không có nhãn -> tự quét thư mục ảnh
            self.image_files = sorted([f for f in os.listdir(self.img_dir)
                                      if f.endswith(('.jpg', '.jpeg', '.png'))])
            self.annotations = pd.DataFrame({'file_name': self.image_files})
        else:
            # Train set có file CSV chứa nhãn
            self.annotations = pd.read_csv(os.path.join(root_dir, csv_file_name))

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        row = self.annotations.iloc[idx]
        img_path = os.path.join(self.img_dir, row['file_name'])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, -1  # Label giả cho test set

        # Trừ 1 để đưa nhãn 1->6 về chuẩn 0->5 của PyTorch CrossEntropyLoss
        label = int(row['category_id']) - 1
        return image, label

print("Đã định nghĩa TrashDataset")


Đã định nghĩa TrashDataset


In [51]:
class SimpleCNN(nn.Module):
    """
    Mạng CNN đơn giản gồm 2 conv block + Global Average Pooling + Fully Connected
    Input: ảnh RGB 224x224
    Output: logits cho 6 classes
    """
    def __init__(self, num_classes=6, in_ch=3):
        super().__init__()
        # Conv block 1: 3 -> 32 channels, MaxPool giảm spatial xuống 1/2
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_ch, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )
        # Conv block 2: 32 -> 64 channels
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        # Global Average Pooling -> vector 64 chiều
        self.gap = nn.AdaptiveAvgPool2d(1)
        # Fully Connected layer cho phân loại
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.gap(x).flatten(1)
        x = self.fc(x)
        return x


In [52]:
# 4. Chuẩn bị DataLoader cho Validation và Test

BATCH_SIZE = 32
VAL_SPLIT = 0.2
root = "/Users/quangmanh/Project/Olympic-AI-Practice-1/data/TACVU2"

# 1. Dataset & DataLoader cho Validation (CÓ NHÃN THẬT từ train.csv để TÍNH ĐIỂM F1 & Score)
full_dataset = TrashDataset(
    root_dir=os.path.join(root, "train"),
    csv_file_name="train.csv",
    transform=val_transform
)

val_size = int(len(full_dataset) * VAL_SPLIT)
train_size = len(full_dataset) - val_size
generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)
print(f"✅ Đã tạo val_loader: {len(val_dataset)} ảnh có nhãn để ĐÁNH GIÁ (Eval)")

# 2. Dataset & DataLoader cho Public Test (KHÔNG CÓ NHÃN - Dùng để DỰ ĐOÁN và nộp bài)
public_test_dataset = TrashDataset(
    root_dir=os.path.join(root, "public_test"),
    transform=val_transform,
    is_test=True
)
public_test_loader = DataLoader(
    public_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)
print(f"✅ Đã tạo public_test_loader: {len(public_test_dataset)} ảnh để INFERENCE nộp bài")

✅ Đã tạo val_loader: 2122 ảnh có nhãn để ĐÁNH GIÁ (Eval)
✅ Đã tạo public_test_loader: 505 ảnh để INFERENCE nộp bài


In [53]:
# 5. Hàm đánh giá Model Instance (Pretrained / PyTorch)

def eval_model_pretrained(model, dataloader, device="mps", min_score=None, max_score=None):
    """
    Đánh giá mô hình instance (timm, torchvision hoặc custom) trên dataloader có nhãn thật
    """
    model.eval()
    model.to(device)
    all_targets = []
    all_predicts = []

    with torch.no_grad():
        for images, targets in dataloader:
            images = images.to(device)
            outputs = model(images)
            predicts = torch.argmax(outputs, dim=1).cpu().numpy()

            all_predicts.extend(predicts)
            all_targets.extend(targets.numpy())

    macro_f1 = f1_score(all_targets, all_predicts, average="macro")

    final_score = None
    if min_score is not None and max_score is not None:
        if macro_f1 > min_score:
            final_score = 100.0 * (macro_f1 - min_score) / (max_score - min_score)
        else:
            final_score = 0.0

    return {
        "macro_f1": float(macro_f1),
        "final_score": float(final_score) if final_score is not None else None
    }

In [54]:
# 6. Hàm đánh giá Checkpoint từ file .pth / .pt

def eval_checkpoint(
    model: torch.nn.Module,
    checkpoint_path: str,
    dataloader,
    device="mps",
    min_score: float = None,
    max_score: float = None
):
    """
    Nạp weights từ checkpoint và đánh giá trên dataloader có nhãn thật
    """
    state_dict = torch.load(checkpoint_path, map_location=device)
    
    if isinstance(state_dict, dict):
        if "model_state_dict" in state_dict:
            state_dict = state_dict["model_state_dict"]
        elif "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]

    clean_state_dict = {
        (k[7:] if k.startswith("module.") else k): v 
        for k, v in state_dict.items()
    }

    model.load_state_dict(clean_state_dict)
    return eval_model_pretrained(model, dataloader, device=device, min_score=min_score, max_score=max_score)

In [57]:
# 7. Đánh giá Baseline và Pretrained Model trên tập VALIDATION (Có nhãn thật)

device = "mps" if torch.mps.is_available() else "cpu"
MIN_SCORE = 0.0      # Điểm F1 gốc của baseline BTC
MAX_SCORE = 0.9543   # Điểm F1 gốc cao nhất hiện tại trên BXH

# --- 1. ĐÁNH GIÁ MÔ HÌNH BASELINE (SimpleCNN) ---
baseline_model = SimpleCNN(num_classes=6)
result_baseline = eval_checkpoint(
    model=baseline_model,
    checkpoint_path=os.path.join(root, "best_model.pth"),
    dataloader=val_loader,  # ✅ Đánh giá trên tập val có nhãn thật
    device=device,
    min_score=MIN_SCORE,
    max_score=MAX_SCORE
)

print("=== KẾT QUẢ BASELINE (SimpleCNN) ===")
print(f"Điểm gốc (Macro F1): {result_baseline['macro_f1'] * 100:.2f}%")
print(f"Điểm chuẩn hóa (Score): {result_baseline['final_score']:.2f}")

# --- 2. ĐÁNH GIÁ MÔ HÌNH PRETRAINED (Ví dụ ResNet18 / EfficientNet) ---
print("\n=== ĐÁNH GIÁ PRETRAINED MODEL (ResNet18) ===")
pretrained_model = timm.create_model("resnet18", pretrained=True, num_classes=6)
result_pretrained = eval_model_pretrained(
    model=pretrained_model,
    dataloader=val_loader,
    device=device,
    min_score=MIN_SCORE,
    max_score=MAX_SCORE
)
print(f"Điểm gốc (Macro F1): {result_pretrained['macro_f1'] * 100:.2f}%")
print(f"Điểm chuẩn hóa (Score): {result_pretrained['final_score']:.2f}")

=== KẾT QUẢ BASELINE (SimpleCNN) ===
Điểm gốc (Macro F1): 60.35%
Điểm chuẩn hóa (Score): 63.24

=== ĐÁNH GIÁ PRETRAINED MODEL (ResNet18) ===
Điểm gốc (Macro F1): 7.30%
Điểm chuẩn hóa (Score): 7.65
